# Rerun 1 (CPU) — frozen pipeline: single-session E1, factorial ablation, LR-only control, and matched learning curves

This notebook answers the frozen-pipeline part of the second-round review (Paper ID 20266676). It runs **on CPU only** (no accelerator, so it does not use GPU quota).

| Block | Review point | Output |
|---|---|---|
| A. Determinism check | R1-3 (session variability) | `determinism_check.csv`, `environment_cpu.json` |
| B. E1 in one session: all 11 original models + 7 new controls, 10 seeds | R1-3, R2-3 | `E1_SemEval_raw.csv`, `E1_Tokopedia_raw.csv`, `selected_params.csv`, `predictions/` |
| C. Procedure sensitivity (fold stratification, weight objective) | R1-4 | `E1_SemEval_sensitivity_raw.csv` |
| D. Frozen learning curve under three budget regimes, nested subsets, 10 seeds | R1-1, R1-2 | `LC_frozen_raw.csv`, `lc_subset_order.csv` |

New controls (R2-3): `E5 LR (C=0.1)`, `E5 LR (weighted)`, `E5 LR (imbalance-aware)`, `E5 XGB (cost-sensitive)`, `E5 Hybrid (C=0.1)`, `E5 Hybrid (weighted)`, `XLM-R LR (imbalance-aware)`. Together with `E5 LR`, `E5 Hybrid` and `E5 Hybrid (imbalance-aware)` they form two 2 × 2 factorials (class weighting × LR regularization), one for the LR-only head and one for the hybrid head.

Learning-curve regimes (Tokopedia):
* **A — label-matched**: the frozen pipeline fits *and* selects (out-of-fold) on the *n* training texts only; no other label is consulted. The fine-tuned comparator in notebook 04 also sees only these *n* labels (internal 80/20 split for early stopping, then refit on all *n*).
* **B — selection-matched**: both methods fit on the *n* training texts and use the fixed 809-text validation partition only for selection (frozen: ensemble weight and class priors; fine-tuning: early stopping).
* **C — fit on n + validation** (supplementary, = old E2): the frozen pipeline fits and selects on *n* + 809 texts.

**Before running on Kaggle**
1. *Add Input*: SemEval-2025 Task 11 data, PRDECT-ID, and the **embedding-cache dataset** of the main run (strongly recommended: identical embeddings make the determinism check meaningful). To resume an interrupted run, also add the output of the previous version of this notebook.
2. *Settings*: Accelerator **None (CPU)**. Internet only needed if the embedding cache is missing.
3. *Save Version → Save & Run All (Commit)*. Outputs go to `/kaggle/working/rerun_frozen` and are zipped at the end. Every block appends to its CSV after each step and skips finished work, so a rerun resumes.

In [1]:
# ============================================================
# 0. INSTALL & IMPORTS
# ============================================================
!pip -q install iterative-stratification

import os, sys, re, time, json, glob, random, warnings, gc, platform, shutil, hashlib, subprocess
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm
from scipy import stats
import joblib

import sklearn
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, accuracy_score
from sklearn.model_selection import StratifiedShuffleSplit, StratifiedKFold
import xgboost as xgb
import transformers
from transformers import AutoTokenizer, AutoModel
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit, MultilabelStratifiedKFold

warnings.filterwarnings("ignore")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NOTEBOOK_T0 = time.time()
print("Python", platform.python_version(), "| sklearn", sklearn.__version__, "| xgboost", xgb.__version__,
      "| numpy", np.__version__, "| CPUs", os.cpu_count(), "| device", DEVICE)

Python 3.13.15 | sklearn 1.6.1 | xgboost 3.4.1 | numpy 2.1.3 | CPUs 4 | device cpu


## 1. Configuration

In [2]:
# ============================================================
# 1. CONFIG
# ============================================================
SMOKE_TEST = bool(int(os.environ.get("MEXBAL_SMOKE", "0")))   # keep 0 on Kaggle

SEEDS        = [42, 7, 13, 77, 101, 202, 303, 404, 505, 606]
LC_SEEDS     = list(SEEDS)                      # learning curve now uses 10 seeds (was 5)
SEED_PRIMARY = 42
LC_SIZES     = [100, 250, 500, 750, 1000, 1500, 2000, 2500, 3000, 3510]

VAL_FRAC, TEST_FRAC = 0.15, 0.20
N_OOF_FOLDS = 5

# ---- procedure (documented for R1-4) -------------------------
FOLD_STRATIFY_BY_LANG = True    # SemEval inner folds stratified on emotions AND language (was: emotions only)
W_OBJECTIVE           = "pooled"  # ensemble weight maximises pooled out-of-fold Macro F1 (as before)
XGB_NTHREAD           = 4       # fixed thread count (was n_jobs=-1)
XGB_INPUT_DECIMALS    = 5       # XGBoost inputs rounded to 1e-5 so that last-digit BLAS/PCA differences between
                                # machines cannot move histogram bin edges (None = off)

# ---- switches ------------------------------------------------
RUN_DETERMINISM = True
RUN_E1          = True
RUN_SENSITIVITY = True
RUN_LC          = True

ENCODERS = {
    "e5":       dict(name="intfloat/multilingual-e5-base", prefix="query: "),
    "xlmr":     dict(name="xlm-roberta-base",              prefix=None),
    "indobert": dict(name="indobenchmark/indobert-base-p1", prefix=None),
}
MAIN_ENCODER = "e5"
MAX_LEN, EMBED_BATCH = 128, 16

# ---- selection grids (identical for all models, unchanged) ----
WEIGHT_GRID    = np.arange(0.0, 1.0001, 0.05)
THRESHOLD_GRID = np.arange(0.15, 0.8501, 0.01)
MIN_PRECISION, MAX_PPR_RATIO, TIE_TOL = 0.20, 3.0, 0.005
ALPHA_GRID, ALPHA_PASSES = np.arange(0.50, 3.0001, 0.05), 3

XGB_COMMON = dict(max_depth=3, min_child_weight=5, learning_rate=0.05, n_estimators=200,
                  subsample=0.8, colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=1.5,
                  tree_method="hist", verbosity=0)
XGB_RUNTIME = {"n_jobs": XGB_NTHREAD}           # changed only inside the determinism check
XGB_PARAMS_BINARY = dict(objective="binary:logistic", eval_metric="logloss", **XGB_COMMON)
XGB_PARAMS_MULTI  = dict(objective="multi:softprob", eval_metric="mlogloss", **XGB_COMMON)

if SMOKE_TEST:
    SEEDS, LC_SEEDS = [42, 7, 13], [42, 7, 13]
    LC_SIZES = [100, 500]

# ---- paths ------------------------------------------------------
INPUT_ROOTS = ("/kaggle/input", os.environ.get("MEXBAL_DATA", "/nonexistent"))

def _find(pattern, roots=INPUT_ROOTS):
    for r in roots:
        hits = sorted(glob.glob(os.path.join(r, "**", pattern), recursive=True))
        if hits: return hits[0]
    return None

def _semeval_root():
    hits = [h for r in INPUT_ROOTS for h in sorted(glob.glob(os.path.join(r, "**", "track_a/train/sun.csv"), recursive=True))]
    hits.sort(key=lambda h: "public_data_test" not in h)
    for h in hits:
        root = Path(h).parents[2]
        need = [root / "track_a/dev/sun.csv", root / "track_c/dev/jav.csv", root / "track_c/dev/ind.csv"]
        if all(f.exists() for f in need) and all({"anger", "joy"} <= set(pd.read_csv(f, nrows=1).columns) for f in need):
            return str(root)
    return None

SEMEVAL_ROOT = _semeval_root()
assert SEMEVAL_ROOT, "SemEval-2025 Task 11 data (with labelled dev files) not found: add the dataset as input."
TOKOPEDIA_CSV = _find("PRDECT-ID Dataset.csv") or _find("*PRDECT*.csv") or _find("Tokopedia Review.csv")
assert TOKOPEDIA_CSV, "PRDECT-ID (Tokopedia) CSV not found: add the dataset as input."

WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.environ.get("MEXBAL_WORK", "./work")
OUT_ROOT  = os.path.join(WORK, "rerun_frozen")
CACHE_OUT = os.path.join(WORK, "embedding_cache")
_cache_hit = _find("tokopedia_e5.npz")
CACHE_IN  = str(Path(_cache_hit).parent) if _cache_hit else None

# resume: copy the outputs of a previous version of this notebook (added as input) into the working folder
_prev = [p for r in INPUT_ROOTS for p in glob.glob(os.path.join(r, "**", "rerun_frozen"), recursive=True) if os.path.isdir(p)]
if _prev and not os.path.exists(OUT_ROOT):
    shutil.copytree(_prev[0], OUT_ROOT); print("Resumed from", _prev[0])
for d in [OUT_ROOT, CACHE_OUT, f"{OUT_ROOT}/predictions"]:
    Path(d).mkdir(parents=True, exist_ok=True)
print("SemEval root :", SEMEVAL_ROOT); print("Tokopedia csv:", TOKOPEDIA_CSV)
print("Cache input  :", CACHE_IN);     print("Output       :", OUT_ROOT, "| SMOKE_TEST =", SMOKE_TEST)

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)

def append_csv(path, rows):
    new = pd.DataFrame(rows)
    old = pd.read_csv(path) if os.path.exists(path) else pd.DataFrame()
    df = pd.concat([old, new], ignore_index=True); df.to_csv(path, index=False); return df

def done_keys(path, cols):
    if not os.path.exists(path): return set()
    df = pd.read_csv(path)
    return set(map(tuple, df[cols].astype(str).values.tolist()))

safe = lambda s: re.sub(r"[^A-Za-z0-9]+", "_", s).strip("_")

SemEval root : /kaggle/input/datasets/aylinnaebzadeh/semeval-2025-task-11-emotion/public_data_test
Tokopedia csv: /kaggle/input/datasets/jocelyndumlao/prdect-id-indonesian-emotion-classification/Product Reviews Dataset for Emotions Classification Tasks - Indonesian (PRDECT-ID) Dataset/PRDECT-ID Dataset.csv
Cache input  : /kaggle/input/datasets/rezkanorhafizah/embedding-cache-me-xbal/embedding_cache
Output       : /kaggle/working/rerun_frozen | SMOKE_TEST = False


## 2. Data, fixed splits, and nested learning-curve subsets

In [3]:
# ============================================================
# 2. DATA (identical to the original notebook) + nested LC order
# ============================================================
SEMEVAL_LABELS = ["anger", "disgust", "fear", "joy", "sadness", "surprise"]
LANGS = ["sun", "jav", "ind"]          # pooling order of the development partitions

def load_semeval():
    sun = pd.concat([pd.read_csv(f"{SEMEVAL_ROOT}/track_a/train/sun.csv"),
                     pd.read_csv(f"{SEMEVAL_ROOT}/track_a/dev/sun.csv")], ignore_index=True)
    jav = pd.read_csv(f"{SEMEVAL_ROOT}/track_c/dev/jav.csv")
    ind = pd.read_csv(f"{SEMEVAL_ROOT}/track_c/dev/ind.csv")
    data = {"sun": sun.reset_index(drop=True), "jav": jav.reset_index(drop=True), "ind": ind.reset_index(drop=True)}
    for lang, df in data.items():
        assert "text" in df.columns and all(c in df.columns for c in SEMEVAL_LABELS), lang
        df["uid"] = [f"{lang}_{i:05d}" for i in range(len(df))]
    return data

def load_tokopedia():
    df = pd.read_csv(TOKOPEDIA_CSV)[["Customer Review", "Emotion"]].dropna().reset_index(drop=True)
    le = LabelEncoder(); df["label"] = le.fit_transform(df["Emotion"])
    df["uid"] = [f"tok_{i:05d}" for i in range(len(df))]
    return df, list(le.classes_)

def _two_stage(splitter, y, seed):
    # identical to the original notebook: 65 / 15 / 20 split, stratified, random_state = seed in both stages
    hold = VAL_FRAC + TEST_FRAC
    tr, h = next(splitter(n_splits=1, test_size=hold, random_state=seed).split(np.zeros(len(y)), y))
    va, te = next(splitter(n_splits=1, test_size=TEST_FRAC / hold, random_state=seed).split(np.zeros(len(h)), y[h]))
    return np.sort(tr), np.sort(h[va]), np.sort(h[te])

LC_ORDER_SALT = 20261023

def lc_order(seed):
    """Nested, approximately stratified ordering of the training partition of one seed.
    The learning-curve subset of size n is the first n entries, so every smaller subset is
    contained in every larger one (nested design). Each class is shuffled and its members are
    spread uniformly over the ranking, so every prefix keeps the class proportions."""
    tr = TOKO_SPLITS[seed][0]; lab = TOKO_DF.label.values[tr]
    rng = np.random.default_rng(LC_ORDER_SALT + seed)
    key = np.empty(len(tr))
    for c in np.unique(lab):
        pos = np.where(lab == c)[0]; m = len(pos)
        key[pos[rng.permutation(m)]] = (np.arange(m) + rng.random(m)) / m
    return np.asarray(tr)[np.argsort(key, kind="stable")]

SEMEVAL_DATA = load_semeval()
TOKO_DF, TOKO_CLASSES = load_tokopedia()
K_TOK = len(TOKO_CLASSES)

ALL_SEEDS = list(dict.fromkeys(SEEDS + LC_SEEDS + [SEED_PRIMARY]))
SEMEVAL_SPLITS = {(l, s): _two_stage(MultilabelStratifiedShuffleSplit, SEMEVAL_DATA[l][SEMEVAL_LABELS].values.astype(int), s)
                  for l in LANGS for s in ALL_SEEDS}
TOKO_SPLITS = {s: _two_stage(StratifiedShuffleSplit, TOKO_DF["label"].values, s) for s in ALL_SEEDS}
for parts in list(SEMEVAL_SPLITS.values()) + list(TOKO_SPLITS.values()):
    tr, va, te = map(set, parts); assert not (tr & va or tr & te or va & te)

# split manifest (same format as the original run, so the two can be compared row by row)
man = []
for (l, s), parts in SEMEVAL_SPLITS.items():
    for sp, idx in zip(["train", "val", "test"], parts):
        man += [{"dataset": "SemEval", "lang": l, "seed": s, "split": sp, "uid": u} for u in SEMEVAL_DATA[l]["uid"].values[idx]]
for s, parts in TOKO_SPLITS.items():
    for sp, idx in zip(["train", "val", "test"], parts):
        man += [{"dataset": "Tokopedia", "lang": "ind", "seed": s, "split": sp, "uid": u} for u in TOKO_DF["uid"].values[idx]]
pd.DataFrame(man).to_csv(f"{OUT_ROOT}/split_manifest_all.csv", index=False)

# nested learning-curve subsets: subset(n) = first n rows of the order of that seed
LC_ORDER = {s: lc_order(s) for s in LC_SEEDS}
rows = []
for s, o in LC_ORDER.items():
    rows += [{"seed": s, "rank": r, "uid": TOKO_DF.uid.values[i], "label": TOKO_CLASSES[TOKO_DF.label.values[i]]} for r, i in enumerate(o)]
pd.DataFrame(rows).to_csv(f"{OUT_ROOT}/lc_subset_order.csv", index=False)
for s in LC_SEEDS[:1]:
    for n in LC_SIZES[:3]:
        print(f"seed {s} n={n} class counts:", np.bincount(TOKO_DF.label.values[LC_ORDER[s][:n]], minlength=K_TOK))
print("SemEval:", {l: len(d) for l, d in SEMEVAL_DATA.items()}, "| Tokopedia:", len(TOKO_DF), TOKO_CLASSES)
print("Tokopedia seed 42 train/val/test:", [len(x) for x in TOKO_SPLITS[42]])

seed 42 n=100 class counts: [13 17 33 15 22]
seed 42 n=250 class counts: [32 43 82 37 56]
seed 42 n=500 class counts: [ 64  86 164  75 111]
SemEval: {'sun': 1123, 'jav': 151, 'ind': 156} | Tokopedia: 5400 ['Anger', 'Fear', 'Happy', 'Love', 'Sadness']
Tokopedia seed 42 train/val/test: [3510, 809, 1081]


## 3. Frozen-encoder embeddings (cached)

In [4]:
# ============================================================
# 3. EMBEDDINGS (identical encoding; reused from the cache when attached)
# ============================================================
def masked_mean_pool(h, mask):
    m = mask.unsqueeze(-1).float()
    return (h * m).sum(1) / m.sum(1).clamp(min=1e-9)

@torch.no_grad()
def encode(texts, model, tok, prefix=None, batch=EMBED_BATCH):
    texts = [(prefix or "") + str(x) for x in texts]; rows = []
    for i in range(0, len(texts), batch):
        enc = tok(texts[i:i + batch], padding=True, truncation=True, max_length=MAX_LEN, return_tensors="pt").to(DEVICE)
        z = masked_mean_pool(model(**enc).last_hidden_state, enc["attention_mask"])
        rows.append(F.normalize(z, p=2, dim=1).cpu().numpy())
    return np.vstack(rows)

def cached_embeddings(cache_name, texts, enc_key):
    for root in [CACHE_IN, CACHE_OUT]:
        if root and os.path.exists(f"{root}/{cache_name}.npz"):
            emb = np.load(f"{root}/{cache_name}.npz")["embedding"]
            if emb.shape[0] == len(texts):
                print("[cache]", cache_name, emb.shape); return emb
    print("[computing on", DEVICE, "]", cache_name, "— this is slow on CPU; attach the embedding cache if possible")
    cfg = ENCODERS[enc_key]
    tok = AutoTokenizer.from_pretrained(cfg["name"]); model = AutoModel.from_pretrained(cfg["name"]).to(DEVICE).eval()
    emb = encode(texts, model, tok, cfg["prefix"])
    np.savez_compressed(f"{CACHE_OUT}/{cache_name}.npz", embedding=emb)
    del model, tok; gc.collect(); return emb

SEMEVAL_EMB = {k: {l: cached_embeddings(f"semeval_{k}_{l}", SEMEVAL_DATA[l]["text"].tolist(), k) for l in LANGS} for k in ENCODERS}
TOKO_EMB = {k: cached_embeddings(f"tokopedia_{k}", TOKO_DF["Customer Review"].tolist(), k) for k in ENCODERS}
EMB_FINGERPRINT = {f"tokopedia_{k}": hashlib.md5(np.round(v, 6).astype(np.float32).tobytes()).hexdigest()[:12] for k, v in TOKO_EMB.items()}
print("Embedding fingerprints:", EMB_FINGERPRINT)

[cache] semeval_e5_sun (1123, 768)
[cache] semeval_e5_jav (151, 768)
[cache] semeval_e5_ind (156, 768)
[cache] semeval_xlmr_sun (1123, 768)
[cache] semeval_xlmr_jav (151, 768)
[cache] semeval_xlmr_ind (156, 768)
[cache] semeval_indobert_sun (1123, 768)
[cache] semeval_indobert_jav (151, 768)
[cache] semeval_indobert_ind (156, 768)
[cache] tokopedia_e5 (5400, 768)
[cache] tokopedia_xlmr (5400, 768)
[cache] tokopedia_indobert (5400, 768)
Embedding fingerprints: {'tokopedia_e5': 'c8b166dc5686', 'tokopedia_xlmr': '12d91e75b8a2', 'tokopedia_indobert': 'cb99cca05bf8'}


## 4. Classifier engines (identical out-of-fold selection for every model; logging of fallbacks)

In [5]:
# ============================================================
# 4. ENGINES
# ============================================================
def fit_preprocessor(X, seed=42):
    sc = StandardScaler().fit(X)
    pc = PCA(n_components=0.95, random_state=seed).fit(sc.transform(X))
    return (lambda A: pc.transform(sc.transform(A))), sc, pc

def xgb_in(Z):
    return np.round(Z, XGB_INPUT_DECIMALS) if XGB_INPUT_DECIMALS is not None else Z

def new_log():
    return {"constant_fits": 0, "threshold_fallbacks": 0}

class ConstantBinary:
    # used when an emotion has no positive (or no negative) example in a training fold
    def __init__(self, p): self.p = float(p)
    def predict_proba(self, X): return np.c_[np.full(len(X), 1 - self.p), np.full(len(X), self.p)]

def combine(px, pl, w):
    if px is None: return pl
    if pl is None: return px
    return w * px + (1 - w) * pl

# ---------------- multi-label (SemEval) ----------------
def train_ml(X, y, cfg, seed, log=None):
    xm = {} if cfg["use_xgb"] else None; lm = {} if cfg["use_lr"] else None
    for j, e in enumerate(SEMEVAL_LABELS):
        yy = y[:, j]; pos = int(yy.sum()); neg = len(yy) - pos
        degenerate = pos == 0 or neg == 0
        if degenerate and log is not None: log["constant_fits"] += 1
        if cfg["use_xgb"]:
            if degenerate: xm[e] = ConstantBinary(yy.mean())
            else:
                ratio = neg / pos if cfg["cost_sensitive"] else 1.0
                xm[e] = xgb.XGBClassifier(scale_pos_weight=ratio, random_state=seed, n_jobs=XGB_RUNTIME["n_jobs"],
                                          **XGB_PARAMS_BINARY).fit(xgb_in(X), yy)
        if cfg["use_lr"]:
            if degenerate: lm[e] = ConstantBinary(yy.mean())
            else:
                lm[e] = LogisticRegression(C=cfg["lr_C"], max_iter=1000, solver="lbfgs",
                                           class_weight="balanced" if cfg["lr_balanced"] else None,
                                           random_state=seed).fit(X, yy)
    return xm, lm

def proba_ml(X, xm, lm):
    px = np.column_stack([xm[e].predict_proba(xgb_in(X))[:, 1] for e in SEMEVAL_LABELS]) if xm is not None else None
    pl = np.column_stack([lm[e].predict_proba(X)[:, 1] for e in SEMEVAL_LABELS]) if lm is not None else None
    return px, pl

def ml_score(y, yp, groups=None, objective="pooled"):
    if objective == "pooled" or groups is None:
        return f1_score(y, yp, average="macro", zero_division=0)
    return float(np.mean([f1_score(y[groups == g], yp[groups == g], average="macro", zero_division=0) for g in np.unique(groups)]))

def tune_w_ml(px, pl, y, groups=None, objective="pooled"):
    # ensemble weight at a fixed 0.5 threshold; ties -> smallest weight (first maximum)
    best = (0.5, -np.inf)
    for w in WEIGHT_GRID:
        s = ml_score(y, (combine(px, pl, w) >= 0.5).astype(int), groups, objective)
        if s > best[1]: best = (float(w), s)
    return best[0]

def tune_th(p, y, log=None):
    # one global threshold per emotion; constraints: precision >= 0.20 and positive rate <= max(3*base, base+0.05);
    # ties within 0.005 -> highest threshold; if no threshold satisfies the constraints -> unconstrained best F1
    out = []
    for j in range(p.shape[1]):
        base = float(y[:, j].mean()); cap = min(1.0, max(MAX_PPR_RATIO * base, base + 0.05)); sc = []
        for t in THRESHOLD_GRID:
            yp = (p[:, j] >= t).astype(int)
            if yp.mean() > cap: sc.append(-np.inf); continue
            prec = y[:, j][yp == 1].mean() if yp.sum() else 0
            if prec < MIN_PRECISION: sc.append(-np.inf); continue
            sc.append(f1_score(y[:, j], yp, zero_division=0))
        a = np.asarray(sc)
        if not np.isfinite(a).any():
            if log is not None: log["threshold_fallbacks"] += 1
            a = np.asarray([f1_score(y[:, j], (p[:, j] >= t).astype(int), zero_division=0) for t in THRESHOLD_GRID])
        out.append(float(THRESHOLD_GRID[np.where(a >= np.nanmax(a) - TIE_TOL)[0][-1]]))
    return np.asarray(out)

def oof_ml(X, y, cfg, seed, strat=None, log=None):
    px = np.zeros(y.shape) if cfg["use_xgb"] else None; pl = np.zeros(y.shape) if cfg["use_lr"] else None
    S = y if strat is None else np.c_[y, strat]
    for a, b in MultilabelStratifiedKFold(n_splits=N_OOF_FOLDS, shuffle=True, random_state=seed).split(np.zeros(len(y)), S):
        tf, _, _ = fit_preprocessor(X[a], seed); xm, lm = train_ml(tf(X[a]), y[a], cfg, seed, log)
        qx, ql = proba_ml(tf(X[b]), xm, lm)
        if px is not None: px[b] = qx
        if pl is not None: pl[b] = ql
    return px, pl

# ---------------- single-label (Tokopedia) ----------------
def train_sl(X, y, cfg, seed):
    xm = lm = None
    if cfg["use_xgb"]:
        sw = None
        if cfg["cost_sensitive"]:
            cnt = np.bincount(y, minlength=K_TOK); wt = len(y) / (K_TOK * np.maximum(cnt, 1)); sw = wt[y]
        xm = xgb.XGBClassifier(num_class=K_TOK, random_state=seed, n_jobs=XGB_RUNTIME["n_jobs"],
                               **XGB_PARAMS_MULTI).fit(xgb_in(X), y, sample_weight=sw)
    if cfg["use_lr"]:
        lm = LogisticRegression(C=cfg["lr_C"], max_iter=1000, solver="lbfgs",
                                class_weight="balanced" if cfg["lr_balanced"] else None, random_state=seed).fit(X, y)
    return xm, lm

def proba_sl(X, xm, lm):
    return (xm.predict_proba(xgb_in(X)) if xm is not None else None), (lm.predict_proba(X) if lm is not None else None)

def oof_sl(X, y, cfg, seed):
    px = np.zeros((len(y), K_TOK)) if cfg["use_xgb"] else None; pl = np.zeros((len(y), K_TOK)) if cfg["use_lr"] else None
    for a, b in StratifiedKFold(n_splits=N_OOF_FOLDS, shuffle=True, random_state=seed).split(np.zeros(len(y)), y):
        tf, _, _ = fit_preprocessor(X[a], seed); xm, lm = train_sl(tf(X[a]), y[a], cfg, seed)
        qx, ql = proba_sl(tf(X[b]), xm, lm)
        if px is not None: px[b] = qx
        if pl is not None: pl[b] = ql
    return px, pl

def tune_w_sl(px, pl, y):
    best = (0.5, -np.inf)
    for w in WEIGHT_GRID:
        s = f1_score(y, np.argmax(combine(px, pl, w), 1), average="macro", zero_division=0)
        if s > best[1]: best = (float(w), s)
    return best[0]

def tune_alpha(p, y):
    # class-prior coefficients by coordinate ascent; ties within 0.005 -> value closest to 1
    alpha = np.ones(K_TOK)
    for _ in range(ALPHA_PASSES):
        for c in range(K_TOK):
            sc = []
            for a in ALPHA_GRID:
                q = alpha.copy(); q[c] = a
                sc.append(f1_score(y, np.argmax(p * q, 1), average="macro", zero_division=0))
            sc = np.asarray(sc); cand = np.where(sc >= sc.max() - TIE_TOL)[0]
            alpha[c] = ALPHA_GRID[cand[np.argmin(np.abs(ALPHA_GRID[cand] - 1))]]
    return alpha

## 5. Model configurations and runners

In [6]:
# ============================================================
# 5. CONFIGURATIONS
# ============================================================
def C(name, encoder, use_xgb=True, use_lr=True, cost_sensitive=False, lr_balanced=False, lr_C=1.0):
    return dict(name=name, encoder=encoder, use_xgb=use_xgb, use_lr=use_lr,
                cost_sensitive=cost_sensitive, lr_balanced=lr_balanced, lr_C=lr_C)

W = dict(cost_sensitive=True, lr_balanced=True)            # class weighting (both branches)
REF = "E5 Hybrid (imbalance-aware)"                        # = ME-XBAL in the old result files
CONFIGS = [
    # --- the 11 models of the manuscript (old labels: ME-XBAL (XLM-R) -> XLM-R Hybrid (imbalance-aware), ME-XBAL -> REF)
    C("XLM-R LR", "xlmr", use_xgb=False), C("XLM-R XGB", "xlmr", use_lr=False), C("XLM-R Hybrid", "xlmr"),
    C("IndoBERT LR", "indobert", use_xgb=False), C("IndoBERT XGB", "indobert", use_lr=False), C("IndoBERT Hybrid", "indobert"),
    C("E5 LR", "e5", use_xgb=False), C("E5 XGB", "e5", use_lr=False), C("E5 Hybrid", "e5"),
    C("XLM-R Hybrid (imbalance-aware)", "xlmr", **W, lr_C=0.1), C(REF, "e5", **W, lr_C=0.1),
    # --- new controls (R2-3, R1-3): 2 x 2 factorials (weighting x C) for the LR-only and the hybrid head
    C("E5 LR (C=0.1)", "e5", use_xgb=False, lr_C=0.1),
    C("E5 LR (weighted)", "e5", use_xgb=False, lr_balanced=True),
    C("E5 LR (imbalance-aware)", "e5", use_xgb=False, lr_balanced=True, lr_C=0.1),
    C("E5 XGB (cost-sensitive)", "e5", use_lr=False, cost_sensitive=True),
    C("E5 Hybrid (C=0.1)", "e5", lr_C=0.1),
    C("E5 Hybrid (weighted)", "e5", **W),
    C("XLM-R LR (imbalance-aware)", "xlmr", use_xgb=False, lr_balanced=True, lr_C=0.1),
]
CFG = {c["name"]: c for c in CONFIGS}
assert len(CFG) == len(CONFIGS)
print(len(CONFIGS), "configurations")

# ============================================================
# 6. RUNNERS
# ============================================================
def lang_onehot(groups):
    return np.eye(len(LANGS), dtype=int)[groups]

def run_sem(cfg, seed, fold_by_lang=None, w_objective=None):
    fold_by_lang = FOLD_STRATIFY_BY_LANG if fold_by_lang is None else fold_by_lang
    w_objective = w_objective or W_OBJECTIVE
    E = SEMEVAL_EMB[cfg["encoder"]]; log = new_log()
    dev = {l: np.r_[SEMEVAL_SPLITS[(l, seed)][0], SEMEVAL_SPLITS[(l, seed)][1]] for l in LANGS}
    X = np.vstack([E[l][dev[l]] for l in LANGS])
    y = np.vstack([SEMEVAL_DATA[l][SEMEVAL_LABELS].values.astype(int)[dev[l]] for l in LANGS])
    g = np.concatenate([np.full(len(dev[l]), i) for i, l in enumerate(LANGS)])
    px, pl = oof_ml(X, y, cfg, seed, lang_onehot(g) if fold_by_lang else None, log)
    w = tune_w_ml(px, pl, y, g, w_objective) if (cfg["use_xgb"] and cfg["use_lr"]) else (1.0 if cfg["use_xgb"] else 0.0)
    th = tune_th(combine(px, pl, w), y, log)
    tf, _, _ = fit_preprocessor(X, seed); xm, lm = train_ml(tf(X), y, cfg, seed, log)
    rows, art, YT, YP = [], {"weight_xgb": w, "thresholds": th, "log": log, "n_dev": {l: len(dev[l]) for l in LANGS}}, [], []
    for l in LANGS:
        te = SEMEVAL_SPLITS[(l, seed)][2]
        yt = SEMEVAL_DATA[l][SEMEVAL_LABELS].values.astype(int)[te]
        pt = combine(*proba_ml(tf(E[l][te]), xm, lm), w); yp = (pt >= th[None, :]).astype(int)
        rows.append({"model": cfg["name"], "seed": seed, "lang": l, "n_test": len(te), "weight_xgb": w,
                     "macro_f1": f1_score(yt, yp, average="macro", zero_division=0)})
        art[l] = {"test_uid": SEMEVAL_DATA[l]["uid"].values[te], "y_true": yt, "y_pred": yp, "proba": pt.astype(np.float32)}
        YT.append(yt); YP.append(yp)
    YT, YP = np.vstack(YT), np.vstack(YP)
    rows.append({"model": cfg["name"], "seed": seed, "lang": "pooled", "n_test": len(YT), "weight_xgb": w,
                 "macro_f1": f1_score(YT, YP, average="macro", zero_division=0)})
    rows.append({"model": cfg["name"], "seed": seed, "lang": "average", "n_test": len(YT), "weight_xgb": w,
                 "macro_f1": float(np.mean([r["macro_f1"] for r in rows[:3]]))})
    for r in rows: r.update(constant_fits=log["constant_fits"], threshold_fallbacks=log["threshold_fallbacks"])
    return rows, art

def _tok_eval(cfg, seed, xm, lm, tf, w, alpha, te):
    E = TOKO_EMB[cfg["encoder"]]; yt = TOKO_DF.label.values[te]
    pt = combine(*proba_sl(tf(E[te]), xm, lm), w); yp = np.argmax(pt * alpha, 1)
    row = {"model": cfg["name"], "seed": seed, "n_test": len(te), "weight_xgb": w,
           "macro_f1": f1_score(yt, yp, average="macro", zero_division=0),
           "micro_f1": f1_score(yt, yp, average="micro", zero_division=0), "accuracy": accuracy_score(yt, yp),
           "alpha": json.dumps([round(float(a), 2) for a in alpha])}
    art = {"weight_xgb": w, "alpha": alpha, "test_uid": TOKO_DF.uid.values[te], "y_true": yt, "y_pred": yp,
           "proba": pt.astype(np.float32)}
    return row, art

def run_tok(cfg, seed, fit_idx=None):
    """Fit AND select (out-of-fold) on fit_idx (default: train + validation = development partition)."""
    E = TOKO_EMB[cfg["encoder"]]; tr, va, te = TOKO_SPLITS[seed]
    dev = np.r_[tr, va] if fit_idx is None else np.asarray(fit_idx)
    X = E[dev]; y = TOKO_DF.label.values[dev]
    px, pl = oof_sl(X, y, cfg, seed)
    w = tune_w_sl(px, pl, y) if (cfg["use_xgb"] and cfg["use_lr"]) else (1.0 if cfg["use_xgb"] else 0.0)
    alpha = tune_alpha(combine(px, pl, w), y)
    tf, _, _ = fit_preprocessor(X, seed); xm, lm = train_sl(tf(X), y, cfg, seed)
    return _tok_eval(cfg, seed, xm, lm, tf, w, alpha, te)

def run_tok_valsel(cfg, seed, fit_idx, sel_idx):
    """Fit on fit_idx only; select the ensemble weight and class priors on sel_idx (regime B)."""
    E = TOKO_EMB[cfg["encoder"]]; te = TOKO_SPLITS[seed][2]
    X = E[fit_idx]; y = TOKO_DF.label.values[fit_idx]; ys = TOKO_DF.label.values[sel_idx]
    tf, _, _ = fit_preprocessor(X, seed); xm, lm = train_sl(tf(X), y, cfg, seed)
    px, pl = proba_sl(tf(E[sel_idx]), xm, lm)
    w = tune_w_sl(px, pl, ys) if (cfg["use_xgb"] and cfg["use_lr"]) else (1.0 if cfg["use_xgb"] else 0.0)
    alpha = tune_alpha(combine(px, pl, w), ys)
    return _tok_eval(cfg, seed, xm, lm, tf, w, alpha, te)

18 configurations


## 6. A — Determinism check (R1-3)

In [7]:
# ============================================================
# 7. DETERMINISM CHECK
#  (a) the same model run twice in this session must give identical predictions;
#  (b) XGBoost with 1 vs XGB_NTHREAD threads;
#  (c) LR-only Tokopedia results vs the original run (identical embeddings + splits => identical LR result);
#  (d) record the CPU so that a later session can be compared.
# ============================================================
ORIGINAL_SEED42_TOK = {"E5 LR": 0.622023, "XLM-R LR": 0.577854, "IndoBERT LR": 0.590687,   # results/main/Tokopedia_final_raw.csv
                       "E5 Hybrid": 0.622258, "ME-XBAL": 0.621734}
DET_CSV = f"{OUT_ROOT}/determinism_check.csv"

def cpu_info():
    try: return subprocess.run(["bash", "-c", "lscpu | grep -E 'Model name|^CPU\\(s\\)'"], capture_output=True, text=True).stdout.strip()
    except Exception: return platform.processor()

if RUN_DETERMINISM and not os.path.exists(DET_CSV):
    rows = []
    for name in [REF, "E5 Hybrid"]:
        a1 = run_tok(CFG[name], SEED_PRIMARY)[1]; a2 = run_tok(CFG[name], SEED_PRIMARY)[1]
        rows.append({"check": f"Tokopedia {name}: same session twice", "identical": bool(np.array_equal(a1["y_pred"], a2["y_pred"])),
                     "max_abs_proba_diff": float(np.abs(a1["proba"] - a2["proba"]).max())})
        s1 = run_sem(CFG[name], SEED_PRIMARY)[1]; s2 = run_sem(CFG[name], SEED_PRIMARY)[1]
        rows.append({"check": f"SemEval {name}: same session twice",
                     "identical": all(np.array_equal(s1[l]["y_pred"], s2[l]["y_pred"]) for l in LANGS),
                     "max_abs_proba_diff": float(max(np.abs(s1[l]["proba"] - s2[l]["proba"]).max() for l in LANGS))})
    a_n = run_tok(CFG["E5 XGB"], SEED_PRIMARY)[1]
    XGB_RUNTIME["n_jobs"] = 1; a_1 = run_tok(CFG["E5 XGB"], SEED_PRIMARY)[1]; XGB_RUNTIME["n_jobs"] = XGB_NTHREAD
    rows.append({"check": f"Tokopedia E5 XGB: n_jobs=1 vs n_jobs={XGB_NTHREAD}", "identical": bool(np.array_equal(a_n["y_pred"], a_1["y_pred"])),
                 "max_abs_proba_diff": float(np.abs(a_n["proba"] - a_1["proba"]).max())})
    for name in ["E5 LR", "XLM-R LR", "IndoBERT LR"]:
        r = run_tok(CFG[name], SEED_PRIMARY)[0]
        rows.append({"check": f"Tokopedia {name} seed 42 vs original run (LR only)", "identical": abs(r["macro_f1"] - ORIGINAL_SEED42_TOK[name]) < 5e-6,
                     "max_abs_proba_diff": np.nan, "this_run": r["macro_f1"], "original": ORIGINAL_SEED42_TOK[name]})
    for name, old in [("E5 Hybrid", "E5 Hybrid"), (REF, "ME-XBAL")]:
        r = run_tok(CFG[name], SEED_PRIMARY)[0]
        rows.append({"check": f"Tokopedia {name} seed 42 vs original run (with XGBoost)", "identical": abs(r["macro_f1"] - ORIGINAL_SEED42_TOK[old]) < 5e-6,
                     "max_abs_proba_diff": np.nan, "this_run": r["macro_f1"], "original": ORIGINAL_SEED42_TOK[old]})
    det = pd.DataFrame(rows); det["cpu"] = cpu_info().replace("\n", " | "); det.to_csv(DET_CSV, index=False)
if os.path.exists(DET_CSV): print(pd.read_csv(DET_CSV).drop(columns=["cpu"]).to_string(index=False))

                                                                       check  identical  max_abs_proba_diff  this_run  original
                   Tokopedia E5 Hybrid (imbalance-aware): same session twice       True                 0.0       NaN       NaN
                     SemEval E5 Hybrid (imbalance-aware): same session twice       True                 0.0       NaN       NaN
                                     Tokopedia E5 Hybrid: same session twice       True                 0.0       NaN       NaN
                                       SemEval E5 Hybrid: same session twice       True                 0.0       NaN       NaN
                                      Tokopedia E5 XGB: n_jobs=1 vs n_jobs=4       True                 0.0       NaN       NaN
                           Tokopedia E5 LR seed 42 vs original run (LR only)      False                 NaN  0.620776  0.622023
                        Tokopedia XLM-R LR seed 42 vs original run (LR only)       True                 

## 7. B — E1 in one session: 18 models × 10 seeds (SemEval and Tokopedia)

In [8]:
# ============================================================
# 8. E1 (resumable) + per-run selected parameters
# ============================================================
SEM_RAW, TOK_RAW = f"{OUT_ROOT}/E1_SemEval_raw.csv", f"{OUT_ROOT}/E1_Tokopedia_raw.csv"
PARAMS_CSV = f"{OUT_ROOT}/selected_params.csv"

def log_params(dataset, name, seed, art, variant="primary"):
    append_csv(PARAMS_CSV, [{"dataset": dataset, "variant": variant, "model": name, "seed": seed,
                             "weight_xgb": art["weight_xgb"],
                             "thresholds": json.dumps([round(float(t), 2) for t in art["thresholds"]]) if "thresholds" in art else "",
                             "alpha": json.dumps([round(float(a), 2) for a in art["alpha"]]) if "alpha" in art else "",
                             "constant_fits": art.get("log", {}).get("constant_fits", ""),
                             "threshold_fallbacks": art.get("log", {}).get("threshold_fallbacks", "")}])

if RUN_E1:
    t0 = time.time()
    for seed in SEEDS:
        sem_done, tok_done = done_keys(SEM_RAW, ["model", "seed"]), done_keys(TOK_RAW, ["model", "seed"])
        for cfg in tqdm(CONFIGS, desc=f"E1 seed {seed}"):
            key = (cfg["name"], str(seed))
            if key not in sem_done:
                rows, art = run_sem(cfg, seed); append_csv(SEM_RAW, rows); log_params("SemEval", cfg["name"], seed, art)
                joblib.dump(art, f"{OUT_ROOT}/predictions/SemEval_{safe(cfg['name'])}_seed{seed}.joblib", compress=3)
            if key not in tok_done:
                row, art = run_tok(cfg, seed); append_csv(TOK_RAW, [row]); log_params("Tokopedia", cfg["name"], seed, art)
                joblib.dump(art, f"{OUT_ROOT}/predictions/Tokopedia_{safe(cfg['name'])}_seed{seed}.joblib", compress=3)
    print(f"E1 finished in {(time.time() - t0) / 60:.1f} min")

df_sem, df_tok = pd.read_csv(SEM_RAW), pd.read_csv(TOK_RAW)
pd.set_option("display.width", 250)
print(df_sem.pivot_table(index="model", columns="lang", values="macro_f1", aggfunc="mean")
      [["sun", "jav", "ind", "average", "pooled"]].reindex(CFG).round(4))
print(df_tok.groupby("model")[["macro_f1", "micro_f1"]].mean().reindex(CFG).round(4))

E1 seed 42:   0%|          | 0/18 [00:00<?, ?it/s]

E1 seed 7:   0%|          | 0/18 [00:00<?, ?it/s]

E1 seed 13:   0%|          | 0/18 [00:00<?, ?it/s]

E1 seed 77:   0%|          | 0/18 [00:00<?, ?it/s]

E1 seed 101:   0%|          | 0/18 [00:00<?, ?it/s]

...

E1 seed 303:   0%|          | 0/18 [00:00<?, ?it/s]

E1 seed 404:   0%|          | 0/18 [00:00<?, ?it/s]

E1 seed 505:   0%|          | 0/18 [00:00<?, ?it/s]

E1 seed 606:   0%|          | 0/18 [00:00<?, ?it/s]

E1 finished in 315.2 min
lang                               sun     jav     ind  average  pooled
model                                                                  
XLM-R LR                        0.3790  0.3793  0.4092   0.3892  0.3933
XLM-R XGB                       0.3550  0.3423  0.3645   0.3539  0.3633
XLM-R Hybrid                    0.3789  0.3814  0.4080   0.3894  0.3934
IndoBERT LR                     0.3780  0.3403  0.3855   0.3679  0.3830
IndoBERT XGB                    0.3562  0.3036  0.3985   0.3528  0.3671
IndoBERT Hybrid                 0.3740  0.3387  0.3805   0.3644  0.3797
E5 LR                           0.4007  0.3957  0.4003   0.3989  0.4115
E5 XGB                          0.4063  0.3536  0.4115   0.3905  0.4073
E5 Hybrid                       0.4026  0.3982  0.3984   0.3997  0.4132
XLM-R Hybrid (imbalance-aware)  0.3946  0.4043  0.4241   0.4076  0.4089
E5 Hybrid (imbalance-aware)     0.4180  0.4336  0.4156   0.4224  0.4295
E5 LR (C=0.1)                   0.4129 

## 8. C — Procedure sensitivity on SemEval (R1-4)

In [9]:
# ============================================================
# 9. SENSITIVITY: (i) inner folds stratified on emotions only (old procedure);
#                 (ii) ensemble weight selected on the language-averaged instead of the pooled Macro F1
# ============================================================
SENS_RAW = f"{OUT_ROOT}/E1_SemEval_sensitivity_raw.csv"
VARIANTS = {"folds: emotions only": dict(fold_by_lang=False, w_objective="pooled"),
            "weight objective: language average": dict(fold_by_lang=FOLD_STRATIFY_BY_LANG, w_objective="lang_avg")}
if RUN_SENSITIVITY:
    done = done_keys(SENS_RAW, ["variant", "model", "seed"])
    for vname, kw in VARIANTS.items():
        for name in [REF, "E5 Hybrid", "E5 LR (imbalance-aware)"]:
            for seed in SEEDS:
                if (vname, name, str(seed)) in done: continue
                rows, art = run_sem(CFG[name], seed, **kw)
                append_csv(SENS_RAW, [dict(r, variant=vname) for r in rows]); log_params("SemEval", name, seed, art, vname)
    s = pd.read_csv(SENS_RAW)
    print(s.pivot_table(index=["variant", "model"], columns="lang", values="macro_f1", aggfunc="mean")[["average", "pooled"]].round(4))

lang                                                            average  pooled
variant                            model                                       
folds: emotions only               E5 Hybrid                     0.3949  0.4072
                                   E5 Hybrid (imbalance-aware)   0.4172  0.4269
                                   E5 LR (imbalance-aware)       0.4262  0.4322
weight objective: language average E5 Hybrid                     0.3991  0.4128
                                   E5 Hybrid (imbalance-aware)   0.4206  0.4275
                                   E5 LR (imbalance-aware)       0.4209  0.4293


## 9. D — Frozen learning curve under three budget regimes (R1-1, R1-2)

In [10]:
# ============================================================
# 10. LEARNING CURVE (Tokopedia, nested subsets, 10 seeds)
#   A_label_matched     : fit + out-of-fold selection on the n training texts only
#   B_selection_matched : fit on the n texts, select weight/priors on the fixed 809-text validation partition
#   C_fit_with_val      : fit + out-of-fold selection on n + validation (supplementary; = old E2)
# ============================================================
LC_RAW = f"{OUT_ROOT}/LC_frozen_raw.csv"
LC_MODELS = [REF, "E5 LR (imbalance-aware)"]
if RUN_LC:
    t0 = time.time()
    for seed in LC_SEEDS:
        done = done_keys(LC_RAW, ["regime", "model", "seed", "train_size"])
        tr_full, va, te = TOKO_SPLITS[seed]
        for n in tqdm(LC_SIZES, desc=f"LC seed {seed}"):
            sub = LC_ORDER[seed][:n]
            for name in LC_MODELS:
                for regime in ["A_label_matched", "B_selection_matched", "C_fit_with_val"]:
                    if (regime, name, str(seed), str(n)) in done: continue
                    t1 = time.perf_counter()
                    if regime == "A_label_matched":       r, _ = run_tok(CFG[name], seed, fit_idx=sub)
                    elif regime == "B_selection_matched": r, _ = run_tok_valsel(CFG[name], seed, sub, va)
                    else:                                  r, _ = run_tok(CFG[name], seed, fit_idx=np.r_[sub, va])
                    r.update(regime=regime, train_size=n, fit_time_s=time.perf_counter() - t1,
                             labels_used=n if regime == "A_label_matched" else n + len(va))
                    append_csv(LC_RAW, [r])
    print(f"LC finished in {(time.time() - t0) / 60:.1f} min")
    lc = pd.read_csv(LC_RAW)
    print(lc.pivot_table(index=["regime", "model"], columns="train_size", values="macro_f1", aggfunc="mean").round(4))

LC seed 42:   0%|          | 0/10 [00:00<?, ?it/s]

LC seed 7:   0%|          | 0/10 [00:00<?, ?it/s]

LC seed 13:   0%|          | 0/10 [00:00<?, ?it/s]

LC seed 77:   0%|          | 0/10 [00:00<?, ?it/s]

LC seed 101:   0%|          | 0/10 [00:00<?, ?it/s]

...

LC seed 303:   0%|          | 0/10 [00:00<?, ?it/s]

LC seed 404:   0%|          | 0/10 [00:00<?, ?it/s]

LC seed 505:   0%|          | 0/10 [00:00<?, ?it/s]

LC seed 606:   0%|          | 0/10 [00:00<?, ?it/s]

LC finished in 325.9 min
train_size                                         100     250     500     750     1000    1500    2000    2500    3000    3510
regime              model                                                                                                      
A_label_matched     E5 Hybrid (imbalance-aware)  0.4679  0.5230  0.5452  0.5564  0.5704  0.5797  0.5936  0.6063  0.6135  0.6165
                    E5 LR (imbalance-aware)      0.4791  0.5232  0.5377  0.5512  0.5620  0.5596  0.5751  0.5871  0.5983  0.6050
B_selection_matched E5 Hybrid (imbalance-aware)  0.4844  0.5294  0.5402  0.5563  0.5663  0.5802  0.5915  0.6034  0.6125  0.6220
                    E5 LR (imbalance-aware)      0.4834  0.5278  0.5373  0.5535  0.5634  0.5600  0.5755  0.5877  0.5978  0.6065
C_fit_with_val      E5 Hybrid (imbalance-aware)  0.5582  0.5657  0.5727  0.5779  0.5893  0.6002  0.6111  0.6198  0.6244  0.6289
                    E5 LR (imbalance-aware)      0.5513  0.5546  0.5548  0.5631

## 10. Environment record and archive

In [11]:
# ============================================================
# 11. ENVIRONMENT + ZIP
# ============================================================
env = {"python": platform.python_version(), "torch": torch.__version__, "transformers": transformers.__version__,
       "sklearn": sklearn.__version__, "xgboost": xgb.__version__, "numpy": np.__version__, "pandas": pd.__version__,
       "cpu": cpu_info(), "cpu_count": os.cpu_count(), "xgb_nthread": XGB_NTHREAD, "xgb_input_decimals": XGB_INPUT_DECIMALS,
       "fold_stratify_by_lang": FOLD_STRATIFY_BY_LANG, "w_objective": W_OBJECTIVE,
       "seeds": SEEDS, "lc_seeds": LC_SEEDS, "lc_sizes": LC_SIZES, "smoke_test": SMOKE_TEST,
       "embedding_fingerprints": EMB_FINGERPRINT, "embedding_cache_input": CACHE_IN,
       "runtime_min": round((time.time() - NOTEBOOK_T0) / 60, 1)}
json.dump(env, open(f"{OUT_ROOT}/environment_cpu.json", "w"), indent=2)
try:
    open(f"{OUT_ROOT}/pip_freeze_cpu.txt", "w").write(subprocess.run([sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True).stdout)
except Exception as e: print("pip freeze failed:", e)
shutil.make_archive(f"{WORK}/rerun_frozen_outputs", "zip", OUT_ROOT)
print(json.dumps(env, indent=2)); print("Archive:", f"{WORK}/rerun_frozen_outputs.zip")

{
  "python": "3.13.15",
  "torch": "2.11.0+cpu",
  "transformers": "5.16.1",
  "sklearn": "1.6.1",
  "xgboost": "3.4.1",
  "numpy": "2.1.3",
  "pandas": "2.3.3",
  "cpu": "CPU(s):                                  4\nModel name:                              Intel(R) Xeon(R) CPU @ 2.20GHz",
  "cpu_count": 4,
  "xgb_nthread": 4,
  "xgb_input_decimals": 5,
  "fold_stratify_by_lang": true,
  "w_objective": "pooled",
  "seeds": [
    42,
    7,
    13,
    77,
    101,
    202,
    303,
    404,
    505,
    606
  ],
  "lc_seeds": [
    42,
    7,
    13,
    77,
    101,
    202,
    303,
    404,
    505,
    606
  ],
  "lc_sizes": [
    100,
    250,
    500,
    750,
    1000,
    1500,
    2000,
    2500,
    3000,
    3510
  ],
  "smoke_test": false,
  "embedding_fingerprints": {
    "tokopedia_e5": "c8b166dc5686",
    "tokopedia_xlmr": "12d91e75b8a2",
    "tokopedia_indobert": "cb99cca05bf8"
  },
  "embedding_cache_input": "/kaggle/input/datasets/rezkanorhafizah/embedding-cache-me-xb